In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import os, json, sys
utils_dir = '../utils/'
from utils.hm_pyfunctions import *

2026-01-07 09:40:08.299344: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-07 09:40:09.735149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767804010.054618   30874 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767804010.165941   30874 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 09:40:10.999505: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
# get config information
with open(os.path.join(utils_dir, "param_info.json"), "r") as f:
    param_info = json.load(f)

universal_param_names = param_info['u_params']
pft_param_names = param_info['pft_params']

In [3]:
### Load NROY samples
indir = '/glade/work/linnia/CLM6-PPE/ctsm53065_wave3/NROY/'
files = np.sort(glob.glob(indir + 'NROY_param_sets_*'))
df_list = [pd.read_csv(f) for f in files]
NROY_samples = pd.concat(df_list, ignore_index=True)

In [ ]:
# Select an Ensemble of Size N for the next wave
# from NROY sets randomly select N unique universal sets and a coresponding random PFT set
# Repeat the subset sampling n times and score based on how close to a latinhypercube the sample is
# select the sample closest to a latin hypercube
# selected_ix is the index of NROY samples coresponding to paramsets used in the next wave. 
n=1000
N=200
selected_ix = subset_NROY(NROY_samples, n, N, universal_param_names)

In [ ]:
# save the sample as csv (param values are still normalized)
wave3_sets = NROY_samples.iloc[selected_ix]
wave3_sets.to_csv('../paramsets/wave4_paramsets.csv')

In [ ]:
# save the sample as a netCDF (param values are still normalized)
default_params = xr.open_dataset(utils_dir+'/default_params_norm.nc')
param_ranges='~/gen_ensembles/gen_paramfiles/clm6ppe_paramranges_10042025.csv'
paramset_csv = '../paramsets/wave4_paramsets.csv'
outfile = '../paramsets/wave4_paramsets.nc'
write_paramset_ds(default_params, param_ranges, paramset_csv, pft_param_names, outfile)